# Notebook 26. Timing-Group Manual Verification Atlas

This notebook is the manual-verification follow-on to `Notebook 25`.

Its review unit is a **catalog timing group**, not a single event. For one selected timing group, it shows:

- timing-group header and timing summary
- event membership table with all events inside the group
- first-look event-peak plots:
  - `925 hPa` signed divergence
  - `850 hPa q × (-omega)` moist-ascent proxy
  - saved quicklook / OLR / satellite panels when available
- broader “forecast-funnel” synoptic context:
  - `300 hPa` isotachs with `300 hPa` geopotential-height contours
  - `500 hPa` geopotential-height contours
  - surface mean sea-level pressure with `850 hPa` temperature-gradient shading
- continuous timing-group evolution from `Notebook 25`
- a timing-group manual-review scaffold row for later note-taking
- a cross-section section reserved for the next implementation layer

## Plotting standard for this notebook

To keep the figures readable and consistent with the earlier professor feedback:

- event-peak shaded fields use **fixed display levels** rather than per-event autoscaling
- signed fields use **fixed symmetric limits around 0**
- one-sided fields use **fixed upper display limits**
- colorbars are placed **below the panels**, not over the map axes
- legends are placed **outside the data region**
- every plotted quantity is labeled with its meteorological name and units

This notebook is designed as the timing-group manual-review atlas that comes **before** the later cross-section refinement pass.


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = os.environ.get("JPCZ_CATALOG_BRANCH", "codex/notebook16-pcolormesh")
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")


def clone_repo_branch():
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )


def sync_repo_branch():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)


if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    clone_repo_branch()
else:
    print("Using existing repo clone:", REPO_DIR)

try:
    sync_repo_branch()
except subprocess.CalledProcessError:
    print("Existing clone could not switch branches cleanly. Re-cloning target branch.")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    clone_repo_branch()
    sync_repo_branch()

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

active_branch = subprocess.run(["git", "-C", REPO_DIR, "branch", "--show-current"], text=True, capture_output=True, check=True).stdout.strip()
print("Working directory:", os.getcwd())
print("Runtime repo branch:", active_branch)


In [ ]:
from pathlib import Path
import shutil

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch, Polygon
import numpy as np
import ipywidgets as widgets
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, OBJECTIVE_SUBTYPE_DOMAIN, BoundingBox
from jpcz_catalog.diagnostics import (
    load_snapshot,
    compute_geopotential_height_field,
    compute_temperature_gradient_magnitude,
    compute_wind_speed_field,
)
from jpcz_catalog.detect import compute_divergence_field
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.objective_regimes import assign_objective_labels_from_thresholds
from jpcz_catalog.satellite import default_modis_layers_for_date

OBJECTIVE_EXPORT_DIR = Path("outputs/verification/objective_coastal_box_regimes")
TIMING_EXPORT_DIR = Path("outputs/verification/objective_regime_timing_and_impact")
EOF_EXPORT_DIR = Path("outputs/verification/objective_subtype_eof_analysis")
REVIEW_EXPORT_DIR = Path("outputs/verification/objective_regime_timing_group_manual_review")
REVIEW_PLOT_DIR = Path("outputs/verification/objective_regime_timing_group_manual_review_plots")
REVIEW_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_PLOT_DIR.mkdir(parents=True, exist_ok=True)

OBJECTIVE_EVENT_METRICS_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_event_metrics.csv"
LOW_LEVEL_STACK_PATH = EOF_EXPORT_DIR / "cleaned_low_level_eof_event_fields.nc"
TIMING_GROUP_REVIEW_SCAFFOLD_PATH = REVIEW_EXPORT_DIR / "objective_timing_group_manual_review_scaffold.csv"
SURFACE_ELEVATION_CACHE_PATH_26 = Path("outputs/verification/objective_subtype_t850_elevation_mask_sensitivity/objective_subtype_surface_elevation.nc")

QUICKLOOK_DIR = Path("outputs/verification/ndjf_event_quicklooks_merged_12h")
OLR_DIR = Path("outputs/verification/ndjf_event_olr_panels_merged_12h")
SATELLITE_DIR = Path("outputs/verification/ndjf_event_satellite_panels_merged_12h")

CONTINUOUS_RUN_TAG = "fullrun_allprofiles_allspells"
CONTINUOUS_SPELL_GAP_HOURS = 36
CONTINUOUS_PADDING_HOURS = 48
CONTINUOUS_TIME_STEP_HOURS = 6
REVIEW_THRESHOLD_PROFILE = "coastal_permissive"
REVIEW_TIMING_GROUP_ID = None
REVIEW_PREFERRED_PATHS = [
    "offshore_to_coastal",
    "coastal_to_offshore",
    "mixed_or_oscillating",
    "offshore_only",
    "coastal_only",
    "weak_only",
]
SYNOPTIC_MAX_TIMES = 3
ERA5_TIME_CHUNK = 24
SAVE_PLOTS = False

COASTAL_STRIP_VERTICES = (
    (133.05, 35.55),
    (136.05, 35.55),
    (139.55, 39.00),
    (139.55, 42.55),
)
JET_CONTEXT_DOMAIN = BoundingBox(lon_min=110.0, lon_max=170.0, lat_min=20.0, lat_max=50.0)
SURFACE_CONTEXT_DOMAIN = OBJECTIVE_SUBTYPE_DOMAIN
JET_ISOTACH_MIN_WIND_SPEED = 30.0
JET_ISOTACH_LEVELS = np.arange(30.0, 90.0 + 5.0, 5.0)
SURFACE_MSLP_CONTOUR_COLOR = "#39ff14"
TEMP_GRAD_850_CONTOUR_LEVELS = np.array([2.0, 4.0, 6.0])
SURFACE_TGRAD_CONTOUR_COLOR = "#ffb000"
LOW_LEVEL_WIND_STRIDE = 4
LOW_LEVEL_QUIVER_SCALE = 230.0
LOW_LEVEL_REFERENCE_VECTOR = 15.0
LOW_LEVEL_DYNAMICS_LEVEL_HPA = 925
LOW_LEVEL_DIVERGENCE_LEVELS = np.arange(-8.0, 8.0 + 1.0, 1.0)
LOW_LEVEL_TERRAIN_LEVELS = np.array([0.0, 250.0, 500.0, 750.0, 1000.0, 1500.0, 2000.0, 2500.0, 3000.0])
LOW_LEVEL_THETA_CONTOUR_STEP = 2.0
LOW_LEVEL_DYNAMICS_WIND_STRIDE = 4
LOW_LEVEL_DYNAMICS_QUIVER_SCALE = 235.0
LOW_LEVEL_DYNAMICS_REFERENCE_VECTOR = 10.0

CONTINUOUS_CANDIDATE_THRESHOLDS_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_candidate_thresholds_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_SPELL_EVENT_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_events_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_SPELL_WINDOW_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_windows_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_LABELED_TIMESERIES_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_labeled_spell_timeseries_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_SUMMARY_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_spell_summary_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"
CONTINUOUS_PROFILE_SUMMARY_PATH = TIMING_EXPORT_DIR / f"objective_regime_continuous_profile_summary_gap{CONTINUOUS_SPELL_GAP_HOURS:02d}h_pad{CONTINUOUS_PADDING_HOURS:02d}h_{CONTINUOUS_RUN_TAG}.csv"

MOISTURE_FIELD = "vertical_moisture_flux_proxy_850_peak"
DIVERGENCE_FIELD = "divergence_925_peak"

EVENT_FIELD_SPECS = [
    {
        "field_name": MOISTURE_FIELD,
        "panel_title": "850 hPa q × (-omega) moist-ascent proxy",
        "colorbar_label": "850 hPa q × (-omega) moist-ascent proxy [1e-3 Pa s^-1]",
        "cmap": "BrBG",
        "level_step": 0.05,
        "minimum_abs_limit": 0.60,
        "extend": "both",
    },
    {
        "field_name": DIVERGENCE_FIELD,
        "panel_title": "925 hPa signed divergence",
        "colorbar_label": "925 hPa signed divergence [1e-5 s^-1]",
        "cmap": "RdBu_r",
        "level_step": 1.0,
        "minimum_abs_limit": 5.0,
        "extend": "both",
    },
]


def finite_field_values(array_like) -> np.ndarray:
    values = np.asarray(array_like, dtype=float)
    return values[np.isfinite(values)]


def robust_abs_limit(values: np.ndarray, upper_q: float = 0.99) -> float:
    finite = finite_field_values(values)
    if finite.size == 0:
        return 1.0
    return float(np.nanquantile(np.abs(finite), upper_q))


def rounded_limit(value: float, step: float, minimum: float | None = None) -> float:
    if not np.isfinite(value) or value <= 0:
        value = step
    rounded = np.ceil(value / step) * step
    if minimum is not None:
        rounded = max(rounded, minimum)
    return float(rounded)


def coordinate_edges(coord_values: np.ndarray) -> np.ndarray:
    coord_values = np.asarray(coord_values, dtype=float)
    if coord_values.ndim != 1:
        raise ValueError("coordinate_edges expects a 1-D coordinate array")
    if coord_values.size == 1:
        delta = 0.5
        return np.array([coord_values[0] - delta, coord_values[0] + delta], dtype=float)
    diffs = np.diff(coord_values)
    edges = np.empty(coord_values.size + 1, dtype=float)
    edges[1:-1] = coord_values[:-1] + diffs / 2.0
    edges[0] = coord_values[0] - diffs[0] / 2.0
    edges[-1] = coord_values[-1] + diffs[-1] / 2.0
    return edges


def draw_scalar_pcolormesh(ax, data_array, cmap_name: str, levels: np.ndarray, extend: str = "both"):
    lat_edges = coordinate_edges(data_array["latitude"].values)
    lon_edges = coordinate_edges(data_array["longitude"].values)
    cmap = mpl.cm.get_cmap(cmap_name, max(len(levels) - 1, 1))
    norm = BoundaryNorm(levels, ncolors=cmap.N, clip=False)
    mesh = ax.pcolormesh(
        lon_edges,
        lat_edges,
        data_array.values,
        cmap=cmap,
        norm=norm,
        shading="flat",
        transform=ccrs.PlateCarree(),
    )
    mesh.cmap.set_under(cmap(0))
    mesh.cmap.set_over(cmap(cmap.N - 1))
    mesh.colorbar_extend = extend
    return mesh

STATE_COLOR_MAP = {
    "offshore_jpcz_core": "#4c78a8",
    "coastal_interaction": "#f58518",
    "mixed_transition": "#9c755f",
    "weak_or_unclear": "#bab0ac",
}


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive cache:", drive_path)
    return True


def load_cached_dataset(path: Path):
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds.load()


def read_csv_with_dates(path: Path, candidate_date_columns: list[str]) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0)
    parse_dates = [column for column in candidate_date_columns if column in header.columns]
    return pd.read_csv(path, parse_dates=parse_dates)


def quicklook_filename(event_index: int, event_peak: pd.Timestamp) -> str:
    return f"{pd.Timestamp(event_peak).strftime('%Y%m%dT%H%M')}_idx{int(event_index):03d}.png"


def satellite_filename(event_index: int, event_peak: pd.Timestamp) -> str | None:
    layers = default_modis_layers_for_date(pd.Timestamp(event_peak).normalize())
    if not layers:
        return None
    layer_slug = (
        layers[0].replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    timestamp = pd.Timestamp(event_peak).strftime("%Y%m%dT%H%M")
    return f"{timestamp}_idx{int(event_index):03d}_{layer_slug}.jpg"


def first_existing_path(local_dir: Path, filename: str | None) -> Path | None:
    if filename is None:
        return None
    local_path = local_dir / filename
    if local_path.exists():
        return local_path
    if PERSIST_OUTPUTS_TO_DRIVE:
        drive_path = Path(DRIVE_OUTPUT_DIR) / local_dir.name / filename
        if drive_path.exists():
            return drive_path
    return None


def configure_map_axis(ax, domain: BoundingBox, title: str | None = None):
    ax.set_extent([domain.lon_min, domain.lon_max, domain.lat_min, domain.lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#efefef", alpha=0.45, zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.55, edgecolor="0.25")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.25, edgecolor="0.55")
    gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.25, alpha=0.35, color="0.78")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    if title:
        ax.set_title(title, fontsize=12, pad=6)
    return ax


def add_region_outlines(ax, *, include_polygon: bool = True, include_coastal: bool = True):
    handles = []
    labels = []
    if include_polygon:
        polygon_artist = Polygon(
            np.array(JPCZ_POLYGON_VERTICES),
            closed=True,
            fill=False,
            linewidth=1.5,
            edgecolor="#3b82c4",
            transform=ccrs.PlateCarree(),
        )
        ax.add_patch(polygon_artist)
        handles.append(polygon_artist)
        labels.append("JPCZ polygon")
    if include_coastal:
        coastal_artist = Polygon(
            np.array(COASTAL_STRIP_VERTICES),
            closed=True,
            fill=False,
            linewidth=1.5,
            edgecolor="#d97728",
            transform=ccrs.PlateCarree(),
        )
        ax.add_patch(coastal_artist)
        handles.append(coastal_artist)
        labels.append("Coastal wedge")
    return handles, labels


def unique_time_specs(items: list[tuple[str, pd.Timestamp | str | None]], *, limit: int | None = None):
    seen = set()
    ordered = []
    for label, value in items:
        if value is None or pd.isna(value):
            continue
        timestamp = pd.Timestamp(value)
        if timestamp in seen:
            continue
        seen.add(timestamp)
        ordered.append((label, timestamp))
        if limit is not None and len(ordered) >= limit:
            break
    return ordered


def rounded_contour_levels(field: xr.DataArray, *, step: float) -> np.ndarray:
    finite = np.asarray(field.values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.array([0.0])
    lower = np.floor(finite.min() / step) * step
    upper = np.ceil(finite.max() / step) * step
    if lower == upper:
        upper = lower + step
    return np.arange(lower, upper + 0.5 * step, step)


print("Notebook 26 plotting setup")
print("- Event-level first-look maps use fixed display-only levels derived from the Notebook 22 event stack")
print(f"- Review threshold profile: {REVIEW_THRESHOLD_PROFILE}")
print(f"- Review timing group: {REVIEW_TIMING_GROUP_ID or 'auto-select'}")
print(f"- Save plots to outputs: {SAVE_PLOTS}")


In [ ]:
paths_to_restore = [
    OBJECTIVE_EVENT_METRICS_PATH,
    LOW_LEVEL_STACK_PATH,
    CONTINUOUS_CANDIDATE_THRESHOLDS_PATH,
    CONTINUOUS_SPELL_EVENT_PATH,
    CONTINUOUS_SPELL_WINDOW_PATH,
    CONTINUOUS_LABELED_TIMESERIES_PATH,
    CONTINUOUS_SUMMARY_PATH,
    CONTINUOUS_PROFILE_SUMMARY_PATH,
    SURFACE_ELEVATION_CACHE_PATH_26,
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

required_paths = {
    "Notebook 22 objective event metrics": OBJECTIVE_EVENT_METRICS_PATH,
    "Notebook 17 low-level event stack": LOW_LEVEL_STACK_PATH,
    "Notebook 25 candidate thresholds": CONTINUOUS_CANDIDATE_THRESHOLDS_PATH,
    "Notebook 25 timing-group membership": CONTINUOUS_SPELL_EVENT_PATH,
    "Notebook 25 timing-group windows": CONTINUOUS_SPELL_WINDOW_PATH,
    "Notebook 25 labeled timing-group time series": CONTINUOUS_LABELED_TIMESERIES_PATH,
    "Notebook 25 timing-group summary": CONTINUOUS_SUMMARY_PATH,
    "Notebook 25 profile summary": CONTINUOUS_PROFILE_SUMMARY_PATH,
}
missing_paths = [f"{label}: {path}" for label, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise RuntimeError("Missing required Notebook 26 inputs:\n- " + "\n- ".join(missing_paths))

objective_df = read_csv_with_dates(
    OBJECTIVE_EVENT_METRICS_PATH,
    [
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
    ],
)
if "event_index" not in objective_df.columns:
    objective_df["event_index"] = objective_df.index.astype(int)

candidate_thresholds_df = pd.read_csv(CONTINUOUS_CANDIDATE_THRESHOLDS_PATH)
catalog_spell_events_df = read_csv_with_dates(
    CONTINUOUS_SPELL_EVENT_PATH,
    [
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
    ],
)
catalog_spell_window_df = read_csv_with_dates(
    CONTINUOUS_SPELL_WINDOW_PATH,
    [
        "spell_start",
        "spell_end",
        "first_event_peak",
        "last_event_peak",
        "first_event_peak_jst",
        "last_event_peak_jst",
        "analysis_start",
        "analysis_end",
    ],
)
continuous_labeled_timeseries_df = read_csv_with_dates(
    CONTINUOUS_LABELED_TIMESERIES_PATH,
    [
        "time",
        "spell_start",
        "spell_end",
        "analysis_start",
        "analysis_end",
        "first_event_peak",
        "last_event_peak",
        "first_event_peak_jst",
        "last_event_peak_jst",
        "first_offshore_support_time",
        "first_coastal_support_time",
    ],
)
continuous_summary_df = read_csv_with_dates(
    CONTINUOUS_SUMMARY_PATH,
    [
        "spell_start",
        "spell_end",
        "analysis_start",
        "analysis_end",
        "first_event_peak",
        "last_event_peak",
        "first_event_peak_jst",
        "last_event_peak_jst",
        "first_offshore_support_time",
        "first_coastal_support_time",
    ],
)
continuous_profile_summary_df = pd.read_csv(CONTINUOUS_PROFILE_SUMMARY_PATH)
low_level_stack_ds = load_cached_dataset(LOW_LEVEL_STACK_PATH)
if low_level_stack_ds is None:
    raise RuntimeError("Unable to restore the low-level event stack from Notebook 17.")

objective_df = add_japan_local_time_columns(objective_df)

EVENT_FIELD_LEVELS = {}
level_rows = []
for spec in EVENT_FIELD_SPECS:
    field_name = spec["field_name"]
    if field_name not in low_level_stack_ds.data_vars:
        raise RuntimeError(f"Low-level event stack is missing required field: {field_name}")
    values = finite_field_values(low_level_stack_ds[field_name])
    limit = rounded_limit(
        robust_abs_limit(values, upper_q=0.99),
        float(spec["level_step"]),
        minimum=float(spec["minimum_abs_limit"]),
    )
    levels = np.arange(-limit, limit + float(spec["level_step"]) * 0.5, float(spec["level_step"]))
    EVENT_FIELD_LEVELS[field_name] = levels
    level_rows.append(
        {
            "field_name": field_name,
            "panel_title": spec["panel_title"],
            "data_min": float(np.nanmin(values)),
            "data_max": float(np.nanmax(values)),
            "level_min": float(levels[0]),
            "level_max": float(levels[-1]),
            "level_step": float(spec["level_step"]),
            "n_bins": int(max(len(levels) - 1, 0)),
        }
    )
EVENT_FIELD_LEVEL_SUMMARY_DF = pd.DataFrame(level_rows)

print("Notebook 26 inputs loaded")
print(f"- Objective events available: {len(objective_df)}")
print(f"- Timing groups available: {catalog_spell_window_df['catalog_spell_id'].nunique()}")
print(f"- Labeled continuous timing-group rows: {len(continuous_labeled_timeseries_df)}")
print(f"- Threshold profiles available: {candidate_thresholds_df['threshold_profile'].nunique()}")
print(f"- Low-level event stack event count: {low_level_stack_ds.sizes.get('event_index', 'unknown')}")
print("\nFixed display levels chosen for the event-level first-look maps")
display(EVENT_FIELD_LEVEL_SUMMARY_DF)


# Interactive Timing-Group Review Dashboard

This is the primary manual-review interface for `Notebook 26`.

Use it to:
- switch threshold profiles
- jump to any catalog timing group by ID
- tab through the events inside that timing group
- jump among the representative synoptic times
- keep all of the plots inside one tabbed review display
- mark reviewed events / reviewed representative times
- write and save timing-group notes back to the scaffold CSV

Cross-section recommendation for this notebook:
- keep the endpoint capture here as typed values for now
- if we later add click-to-place endpoints, that will be more stable as a **separate dedicated figure step** rather than inside the tabbed dashboard


In [ ]:

required_globals = [
    "objective_df",
    "candidate_thresholds_df",
    "catalog_spell_events_df",
    "catalog_spell_window_df",
    "continuous_labeled_timeseries_df",
    "continuous_summary_df",
    "continuous_profile_summary_df",
    "low_level_stack_ds",
    "EVENT_FIELD_SPECS",
    "EVENT_FIELD_LEVELS",
    "STATE_COLOR_MAP",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the earlier Notebook 26 setup cells first. Missing globals: {missing_globals}")

MANUAL_REVIEW_DEFAULTS = {
    "manual_transition_verified": "",
    "manual_transition_type": "",
    "manual_transition_confidence": "",
    "manual_offshore_onset_jst": "",
    "manual_coastal_onset_jst": "",
    "manual_transition_lag_hours": "",
    "satellite_supports_transition": "",
    "forecast_funnel_supports_transition": "",
    "reviewed_event_indices": "",
    "reviewed_time_roles": "",
    "cross_section_priority": "",
    "cross_section_start_lon": "",
    "cross_section_start_lat": "",
    "cross_section_end_lon": "",
    "cross_section_end_lat": "",
    "coastal_impact_visible": "",
    "coastal_impact_region": "",
    "manual_notes": "",
}
MANUAL_REVIEW_COLUMNS = list(MANUAL_REVIEW_DEFAULTS)
PROFILE_ORDER = candidate_thresholds_df["threshold_profile"].tolist()

EVENT_SEQUENCE_SUMMARY_DF = (
    catalog_spell_events_df
    .sort_values(["catalog_spell_id", "event_peak"])
    .groupby("catalog_spell_id", sort=False)
    .agg(
        grouped_event_count=("event_index", "size"),
        event_indices=("event_index", lambda values: ", ".join(str(int(value)) for value in values)),
        event_level_sequence=("objective_regime_label", lambda values: " -> ".join(str(value) for value in values)),
    )
    .reset_index()
)


def _normalize_text(value, default=""):
    if value is None:
        return default
    if isinstance(value, float) and np.isnan(value):
        return default
    text = str(value).strip()
    if text.lower() == "nan":
        return default
    return text


def _split_csv_tokens(value: str) -> set[str]:
    text = _normalize_text(value)
    if not text:
        return set()
    return {token.strip() for token in text.split(",") if token.strip()}


def _split_pipe_tokens(value: str) -> set[str]:
    text = _normalize_text(value)
    if not text:
        return set()
    return {token.strip() for token in text.split("|") if token.strip()}


def _save_review_scaffold(df: pd.DataFrame):
    df.to_csv(TIMING_GROUP_REVIEW_SCAFFOLD_PATH, index=False)
    maybe_copy_to_drive(TIMING_GROUP_REVIEW_SCAFFOLD_PATH, verbose=False)


def build_or_load_review_scaffold() -> pd.DataFrame:
    scaffold_df = continuous_summary_df.merge(
        EVENT_SEQUENCE_SUMMARY_DF,
        on="catalog_spell_id",
        how="left",
    ).copy()

    for column_name, default_value in MANUAL_REVIEW_DEFAULTS.items():
        scaffold_df[column_name] = default_value

    if TIMING_GROUP_REVIEW_SCAFFOLD_PATH.exists():
        existing_df = pd.read_csv(TIMING_GROUP_REVIEW_SCAFFOLD_PATH, dtype=str).fillna("")
        existing_cols = [
            column_name
            for column_name in ["catalog_spell_id", "threshold_profile", *MANUAL_REVIEW_COLUMNS]
            if column_name in existing_df.columns
        ]
        if {"catalog_spell_id", "threshold_profile"}.issubset(existing_cols):
            scaffold_df = scaffold_df.merge(
                existing_df[existing_cols].drop_duplicates(["catalog_spell_id", "threshold_profile"]),
                on=["catalog_spell_id", "threshold_profile"],
                how="left",
                suffixes=("", "_existing"),
            )
            for column_name in MANUAL_REVIEW_COLUMNS:
                existing_name = f"{column_name}_existing"
                if existing_name in scaffold_df.columns:
                    existing_values = scaffold_df[existing_name].map(_normalize_text)
                    replacement_mask = existing_values != ""
                    scaffold_df.loc[replacement_mask, column_name] = existing_values.loc[replacement_mask]
                    scaffold_df = scaffold_df.drop(columns=[existing_name])

    scaffold_df["threshold_profile"] = pd.Categorical(
        scaffold_df["threshold_profile"],
        categories=PROFILE_ORDER,
        ordered=True,
    )
    scaffold_df = scaffold_df.sort_values(["threshold_profile", "first_event_peak", "catalog_spell_id"]).reset_index(drop=True)
    scaffold_df["threshold_profile"] = scaffold_df["threshold_profile"].astype(str)
    _save_review_scaffold(scaffold_df)
    return scaffold_df


timing_group_review_scaffold_df = build_or_load_review_scaffold()


def build_group_option_label(row: pd.Series) -> str:
    if pd.notna(row.get("first_event_peak_jst")):
        display_time = pd.Timestamp(row["first_event_peak_jst"])
    elif pd.notna(row.get("first_event_peak")):
        display_time = pd.Timestamp(row["first_event_peak"]) + pd.Timedelta(hours=9)
    else:
        display_time = pd.NaT
    date_label = display_time.strftime("%Y-%m-%d") if pd.notna(display_time) else "no-date"
    return (
        f"{row['catalog_spell_id']} | {row['continuous_spell_regime_path']} | "
        f"{int(row['event_count'])} ev | {date_label}"
    )


def get_profile_subset(profile: str) -> pd.DataFrame:
    subset = continuous_summary_df.loc[
        continuous_summary_df["threshold_profile"] == profile
    ].copy()
    subset = subset.sort_values(["first_event_peak", "catalog_spell_id"]).reset_index(drop=True)
    return subset


def choose_default_group_id(profile: str) -> str:
    profile_subset_df = get_profile_subset(profile)
    if profile_subset_df.empty:
        raise RuntimeError(f"No timing-group summaries were found for profile {profile!r}.")

    if REVIEW_TIMING_GROUP_ID is not None:
        review_match = profile_subset_df.loc[
            profile_subset_df["catalog_spell_id"].astype(str) == str(REVIEW_TIMING_GROUP_ID)
        ]
        if not review_match.empty:
            return str(review_match.iloc[0]["catalog_spell_id"])

    selected_group_row = None
    for preferred_path in REVIEW_PREFERRED_PATHS:
        subset = profile_subset_df.loc[
            profile_subset_df["continuous_spell_regime_path"] == preferred_path
        ]
        if not subset.empty:
            selected_group_row = subset.sort_values(
                ["event_count", "analysis_window_hours"],
                ascending=[False, False],
            ).iloc[0]
            break
    if selected_group_row is None:
        selected_group_row = profile_subset_df.sort_values(
            ["event_count", "analysis_window_hours"],
            ascending=[False, False],
        ).iloc[0]
    return str(selected_group_row["catalog_spell_id"])


def build_group_options(profile: str) -> list[tuple[str, str]]:
    profile_subset_df = get_profile_subset(profile)
    return [
        (build_group_option_label(row), str(row["catalog_spell_id"]))
        for _, row in profile_subset_df.iterrows()
    ]


def get_group_context(profile: str, group_id: str) -> dict:
    selected_threshold_row = candidate_thresholds_df.loc[
        candidate_thresholds_df["threshold_profile"] == profile
    ]
    if selected_threshold_row.empty:
        raise RuntimeError(f"Threshold profile {profile!r} was not found in the Notebook 25 threshold table.")
    selected_threshold_row = selected_threshold_row.iloc[0]

    profile_subset_df = get_profile_subset(profile)
    selected_group_row = profile_subset_df.loc[
        profile_subset_df["catalog_spell_id"].astype(str) == str(group_id)
    ]
    if selected_group_row.empty:
        raise RuntimeError(f"Catalog timing group {group_id!r} was not found for profile {profile!r}.")
    selected_group_row = selected_group_row.iloc[0]

    selected_group_events_df = catalog_spell_events_df.loc[
        catalog_spell_events_df["catalog_spell_id"].astype(str) == str(group_id)
    ].sort_values("event_peak").reset_index(drop=True)
    selected_group_window_row = catalog_spell_window_df.loc[
        catalog_spell_window_df["catalog_spell_id"].astype(str) == str(group_id)
    ].iloc[0]
    selected_group_timeseries_df = continuous_labeled_timeseries_df.loc[
        (continuous_labeled_timeseries_df["threshold_profile"] == profile)
        & (continuous_labeled_timeseries_df["catalog_spell_id"].astype(str) == str(group_id))
    ].sort_values("time").reset_index(drop=True)

    selected_group_all_profiles_df = continuous_summary_df.loc[
        continuous_summary_df["catalog_spell_id"].astype(str) == str(group_id)
    ].copy()
    selected_group_all_profiles_df["threshold_profile"] = pd.Categorical(
        selected_group_all_profiles_df["threshold_profile"],
        categories=PROFILE_ORDER,
        ordered=True,
    )
    selected_group_all_profiles_df = selected_group_all_profiles_df.sort_values("threshold_profile").reset_index(drop=True)

    objective_metric_columns = [
        "event_index",
        "objective_regime_label",
        "polygon_qflux_850_mean",
        "coastal_qflux_850_mean",
        "polygon_div_925_mean",
        "coastal_div_925_mean",
    ]
    selected_profile_event_df = (
        selected_group_events_df.drop(columns=["objective_regime_label"], errors="ignore")
        .merge(
            objective_df[[column for column in objective_metric_columns if column in objective_df.columns]],
            on="event_index",
            how="left",
        )
        .rename(columns={"objective_regime_label": "default_threshold_label"})
    )
    selected_profile_event_df = assign_objective_labels_from_thresholds(
        selected_profile_event_df.copy(),
        polygon_qflux_min=float(selected_threshold_row["polygon_qflux_min"]),
        polygon_div_max=float(selected_threshold_row["polygon_div_max"]),
        coastal_qflux_split=float(selected_threshold_row["coastal_qflux_split"]),
        coastal_div_max=float(selected_threshold_row["coastal_div_max"]),
    ).rename(columns={"objective_regime_label": "selected_profile_event_label"})

    representative_time_specs = unique_time_specs(
        [
            ("first event peak", selected_group_row.get("first_event_peak")),
            ("first offshore-support onset", selected_group_row.get("first_offshore_support_time")),
            ("first coastal-support onset", selected_group_row.get("first_coastal_support_time")),
            ("last event peak", selected_group_row.get("last_event_peak")),
        ],
        limit=SYNOPTIC_MAX_TIMES,
    )
    representative_time_table_df = pd.DataFrame(
        {
            "time_role": [label for label, _ in representative_time_specs],
            "time_utc": [timestamp for _, timestamp in representative_time_specs],
        }
    )
    if not representative_time_table_df.empty:
        representative_time_table_df["time_jst"] = representative_time_table_df["time_utc"] + pd.Timedelta(hours=9)

    threshold_display_df = pd.DataFrame(
        {
            "threshold_profile": [profile],
            "polygon_qflux_min": [float(selected_threshold_row["polygon_qflux_min"])],
            "coastal_qflux_split": [float(selected_threshold_row["coastal_qflux_split"])],
            "polygon_div_max": [float(selected_threshold_row["polygon_div_max"])],
            "coastal_div_max": [float(selected_threshold_row["coastal_div_max"])],
        }
    )

    selected_group_header_df = pd.DataFrame(
        [
            {
                "catalog_spell_id": str(group_id),
                "threshold_profile": profile,
                "continuous_spell_regime_path": selected_group_row["continuous_spell_regime_path"],
                "continuous_clear_sequence": selected_group_row["continuous_clear_sequence"],
                "event_count": int(selected_group_row["event_count"]),
                "spell_start": selected_group_row["spell_start"],
                "spell_end": selected_group_row["spell_end"],
                "analysis_start": selected_group_row["analysis_start"],
                "analysis_end": selected_group_row["analysis_end"],
                "first_event_peak_jst": selected_group_row.get("first_event_peak_jst", pd.NaT),
                "last_event_peak_jst": selected_group_row.get("last_event_peak_jst", pd.NaT),
                "first_offshore_support_time": selected_group_row.get("first_offshore_support_time", pd.NaT),
                "first_coastal_support_time": selected_group_row.get("first_coastal_support_time", pd.NaT),
                "offshore_to_coastal_lag_hours": selected_group_row.get("offshore_to_coastal_lag_hours", np.nan),
                "coastal_to_offshore_lag_hours": selected_group_row.get("coastal_to_offshore_lag_hours", np.nan),
                "spell_duration_hours": float(selected_group_row.get("spell_duration_hours", np.nan)),
                "analysis_window_hours": float(selected_group_row.get("analysis_window_hours", np.nan)),
            }
        ]
    )

    selected_event_display_df = selected_profile_event_df[[
        column for column in [
            "event_index",
            "event_start",
            "event_end",
            "event_peak",
            "event_peak_jst",
            "default_threshold_label",
            "selected_profile_event_label",
            "offshore_rule_pass",
            "coastal_rule_pass",
            "polygon_qflux_850_mean",
            "coastal_qflux_850_mean",
            "polygon_div_925_mean",
            "coastal_div_925_mean",
            "gap_from_previous_event_hours",
        ]
        if column in selected_profile_event_df.columns
    ]].copy()

    scaffold_row = timing_group_review_scaffold_df.loc[
        (timing_group_review_scaffold_df["catalog_spell_id"].astype(str) == str(group_id))
        & (timing_group_review_scaffold_df["threshold_profile"] == profile)
    ]
    if scaffold_row.empty:
        raise RuntimeError(f"No scaffold row found for timing group {group_id!r} and profile {profile!r}.")
    scaffold_row = scaffold_row.iloc[0].copy()

    return {
        "threshold_profile": profile,
        "selected_threshold_row": selected_threshold_row,
        "selected_group_id": str(group_id),
        "selected_group_row": selected_group_row,
        "selected_group_events_df": selected_group_events_df,
        "selected_group_window_row": selected_group_window_row,
        "selected_group_timeseries_df": selected_group_timeseries_df,
        "selected_group_all_profiles_df": selected_group_all_profiles_df,
        "selected_profile_event_df": selected_profile_event_df,
        "selected_event_display_df": selected_event_display_df,
        "selected_group_header_df": selected_group_header_df,
        "threshold_display_df": threshold_display_df,
        "representative_time_specs": representative_time_specs,
        "representative_time_table_df": representative_time_table_df,
        "scaffold_row": scaffold_row,
    }


def get_event_row(context: dict, event_index: int) -> pd.Series:
    event_row = context["selected_profile_event_df"].loc[
        context["selected_profile_event_df"]["event_index"].astype(int) == int(event_index)
    ]
    if event_row.empty:
        raise RuntimeError(f"Event index {event_index} was not found in timing group {context['selected_group_id']!r}.")
    return event_row.iloc[0]


def plot_event_firstlook_figure(context: dict, event_index: int):
    event_row = get_event_row(context, event_index)
    event_peak = pd.Timestamp(event_row["event_peak"])
    event_peak_jst = pd.Timestamp(event_row["event_peak_jst"]) if pd.notna(event_row["event_peak_jst"]) else pd.NaT
    event_ds = low_level_stack_ds.sel(event_index=int(event_index))

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(15.4, 5.8),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    fig.subplots_adjust(top=0.82, bottom=0.24, left=0.05, right=0.98, wspace=0.08)

    legend_handles = None
    legend_labels = None
    for ax, spec in zip(axes, EVENT_FIELD_SPECS):
        field = event_ds[spec["field_name"]]
        mesh = draw_scalar_pcolormesh(
            ax,
            field,
            cmap_name=spec["cmap"],
            levels=EVENT_FIELD_LEVELS[spec["field_name"]],
            extend=spec["extend"],
        )
        configure_map_axis(ax, OBJECTIVE_SUBTYPE_DOMAIN, title=spec["panel_title"])
        legend_handles, legend_labels = add_region_outlines(ax)
        bbox = ax.get_position()
        cbar_ax = fig.add_axes([bbox.x0, 0.10, bbox.width, 0.038])
        cbar = fig.colorbar(mesh, cax=cbar_ax, orientation="horizontal")
        cbar.set_label(spec["colorbar_label"], labelpad=4)

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.92),
            ncol=2,
            frameon=False,
        )
    fig.suptitle(
        f"Timing group {context['selected_group_id']} | event {int(event_index):03d} | peak {event_peak:%Y-%m-%d %H:%M UTC}"
        + (f" / {event_peak_jst:%Y-%m-%d %H:%M JST}" if pd.notna(event_peak_jst) else "")
        + f"\ndefault label: {event_row['default_threshold_label']} | {context['threshold_profile']}: {event_row['selected_profile_event_label']}",
        fontsize=13,
        y=0.98,
    )

    if SAVE_PLOTS:
        output_path = REVIEW_PLOT_DIR / f"{context['selected_group_id']}_event_{int(event_index):03d}_firstlook_maps.png"
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        maybe_copy_to_drive(output_path, verbose=False)
    return fig


def render_saved_imagery_for_event(context: dict, event_index: int):
    event_row = get_event_row(context, event_index)
    event_peak = pd.Timestamp(event_row["event_peak"])
    event_peak_jst = pd.Timestamp(event_row["event_peak_jst"]) if pd.notna(event_row["event_peak_jst"]) else pd.NaT

    header_md = f"## Timing group `{context['selected_group_id']}` | event `{int(event_index):03d}`\n\nPeak: `{event_peak:%Y-%m-%d %H:%M UTC}`"
    if pd.notna(event_peak_jst):
        header_md += f" / `{event_peak_jst:%Y-%m-%d %H:%M JST}`"
    header_md += f"\n\nDefault label: `{event_row['default_threshold_label']}`\n\n{context['threshold_profile']} label: `{event_row['selected_profile_event_label']}`"
    display(Markdown(header_md))

    quicklook_path = first_existing_path(QUICKLOOK_DIR, quicklook_filename(int(event_index), event_peak))
    olr_path = first_existing_path(OLR_DIR, quicklook_filename(int(event_index), event_peak))
    satellite_path = first_existing_path(SATELLITE_DIR, satellite_filename(int(event_index), event_peak))

    found_any_panel = False
    for title, panel_path in [
        ("Convergence quicklook", quicklook_path),
        ("OLR / cloud-proxy panel", olr_path),
        ("MODIS daily satellite panel", satellite_path),
    ]:
        if panel_path is None:
            print(f"{title}: not found locally or in Drive cache for event {int(event_index):03d}.")
            continue
        found_any_panel = True
        display(Markdown(f"**{title}**\n\n`{panel_path}`"))
        display(Image(filename=str(panel_path), width=700))

    if not found_any_panel:
        print(f"No quicklook / OLR / satellite panels were found for event {int(event_index):03d}.")


ERA5_RUNTIME_DS_26 = None


def get_era5_runtime_ds():
    global ERA5_RUNTIME_DS_26
    if ERA5_RUNTIME_DS_26 is None:
        ERA5_RUNTIME_DS_26 = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
    return ERA5_RUNTIME_DS_26


SURFACE_ELEVATION_RUNTIME_26 = None


def get_surface_elevation_field_26():
    global SURFACE_ELEVATION_RUNTIME_26
    if SURFACE_ELEVATION_RUNTIME_26 is not None:
        return SURFACE_ELEVATION_RUNTIME_26
    if not SURFACE_ELEVATION_CACHE_PATH_26.exists():
        restore_from_drive_cache(SURFACE_ELEVATION_CACHE_PATH_26)
    terrain_ds = load_cached_dataset(SURFACE_ELEVATION_CACHE_PATH_26)
    if terrain_ds is None or "surface_elevation_m" not in terrain_ds.data_vars:
        return None
    terrain_field = terrain_ds["surface_elevation_m"].astype(float)
    if "longitude" in terrain_field.coords and float(terrain_field.longitude.min().values) < 0.0:
        terrain_field = terrain_field.assign_coords(
            longitude=((terrain_field.longitude + 360.0) % 360.0)
        ).sortby("longitude")
    SURFACE_ELEVATION_RUNTIME_26 = terrain_field
    return SURFACE_ELEVATION_RUNTIME_26


def align_field_to_target(field: xr.DataArray, target_field: xr.DataArray) -> xr.DataArray:
    return field.interp(longitude=target_field.longitude, latitude=target_field.latitude, method="linear")


def compute_potential_temperature_field(
    snapshot: xr.Dataset,
    *,
    temperature_name: str = "temperature",
    pressure_hpa: float | None = None,
) -> xr.DataArray:
    if pressure_hpa is None:
        if "level" not in snapshot.coords:
            raise RuntimeError("A pressure level is required to compute potential temperature.")
        pressure_hpa = float(np.asarray(snapshot["level"]).item())
    kappa = 287.05 / 1004.0
    theta = snapshot[temperature_name] * (1000.0 / float(pressure_hpa)) ** kappa
    theta = theta.rename("potential_temperature")
    theta.attrs["units"] = "K"
    theta.attrs["display_units"] = "K"
    return theta


def plot_synoptic_context_figure(context: dict, time_role: str):
    role_lookup = {label: pd.Timestamp(timestamp) for label, timestamp in context["representative_time_specs"]}
    if time_role not in role_lookup:
        raise RuntimeError(f"Representative time role {time_role!r} is not available for timing group {context['selected_group_id']!r}.")
    analysis_time = role_lookup[time_role]
    era5_runtime_ds = get_era5_runtime_ds()

    snapshot_300 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["u_component_of_wind", "v_component_of_wind", "geopotential"],
        domain=JET_CONTEXT_DOMAIN,
        level=300,
    )
    snapshot_500 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["geopotential"],
        domain=JET_CONTEXT_DOMAIN,
        level=500,
    )
    surface_snapshot_jet = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["mean_sea_level_pressure"],
        domain=JET_CONTEXT_DOMAIN,
    )
    snapshot_850 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["temperature", "u_component_of_wind", "v_component_of_wind"],
        domain=SURFACE_CONTEXT_DOMAIN,
        level=850,
    )
    surface_snapshot_jet = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["mean_sea_level_pressure"],
        domain=JET_CONTEXT_DOMAIN,
    )
    surface_snapshot_local = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["mean_sea_level_pressure"],
        domain=SURFACE_CONTEXT_DOMAIN,
    )

    wind_speed_300 = compute_wind_speed_field(snapshot_300)
    jet_isotach_field = wind_speed_300.where(wind_speed_300 >= JET_ISOTACH_MIN_WIND_SPEED)
    z300 = compute_geopotential_height_field(snapshot_300)
    z500 = compute_geopotential_height_field(snapshot_500)
    temp_grad_850 = compute_temperature_gradient_magnitude(snapshot_850) * 1e5
    temp_grad_850 = temp_grad_850.rename("temperature_gradient_display")
    msl_hpa_jet = (surface_snapshot_jet["mean_sea_level_pressure"] / 100.0).rename("msl_hpa_jet")
    msl_hpa_local = (surface_snapshot_local["mean_sea_level_pressure"] / 100.0).rename("msl_hpa_local")

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18.2, 6.1),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    fig.subplots_adjust(top=0.84, bottom=0.20, left=0.04, right=0.99, wspace=0.08)

    ax = axes[0]
    jet_fill_300 = ax.contourf(
        jet_isotach_field.longitude,
        jet_isotach_field.latitude,
        jet_isotach_field,
        levels=JET_ISOTACH_LEVELS,
        cmap="YlGnBu_r",
        extend="max",
        transform=ccrs.PlateCarree(),
    )
    configure_map_axis(
        ax,
        JET_CONTEXT_DOMAIN,
        title=f"300 hPa jet isotachs (>= {JET_ISOTACH_MIN_WIND_SPEED:.0f} m s^-1) with z300 contours",
    )
    z300_contours = ax.contour(
        z300.longitude,
        z300.latitude,
        z300,
        levels=rounded_contour_levels(z300, step=120.0),
        colors="black",
        linewidths=0.8,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(z300_contours, inline=True, fontsize=7, fmt="%d")
    add_region_outlines(ax)

    ax = axes[1]
    jet_fill_500 = ax.contourf(
        jet_isotach_field.longitude,
        jet_isotach_field.latitude,
        jet_isotach_field,
        levels=JET_ISOTACH_LEVELS,
        cmap="YlGnBu_r",
        extend="max",
        transform=ccrs.PlateCarree(),
    )
    configure_map_axis(ax, JET_CONTEXT_DOMAIN, title="500 hPa geopotential height with 300 hPa jet isotachs and surface MSLP")
    z500_contours = ax.contour(
        z500.longitude,
        z500.latitude,
        z500,
        levels=rounded_contour_levels(z500, step=60.0),
        colors="#303030",
        linewidths=0.9,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(z500_contours, inline=True, fontsize=7, fmt="%d")
    msl_jet_contours = ax.contour(
        msl_hpa_jet.longitude,
        msl_hpa_jet.latitude,
        msl_hpa_jet,
        levels=rounded_contour_levels(msl_hpa_jet, step=4.0),
        colors=SURFACE_MSLP_CONTOUR_COLOR,
        linewidths=0.9,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(msl_jet_contours, inline=True, fontsize=6.5, fmt="%d")
    add_region_outlines(ax)

    ax = axes[2]
    ax.set_extent(
        [SURFACE_CONTEXT_DOMAIN.lon_min, SURFACE_CONTEXT_DOMAIN.lon_max, SURFACE_CONTEXT_DOMAIN.lat_min, SURFACE_CONTEXT_DOMAIN.lat_max],
        crs=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#111111", alpha=0.12, edgecolor="none", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.65, edgecolor="black")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.30, edgecolor="black")
    gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.25, alpha=0.35, color="0.65")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    ax.set_title("Surface MSLP, 850 hPa |grad T| contours, and 850 hPa wind", fontsize=12, pad=6)
    msl_contours = ax.contour(
        msl_hpa_local.longitude,
        msl_hpa_local.latitude,
        msl_hpa_local,
        levels=rounded_contour_levels(msl_hpa_local, step=4.0),
        colors=SURFACE_MSLP_CONTOUR_COLOR,
        linewidths=1.2,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(msl_contours, inline=True, fontsize=7, fmt="%d")
    tempgrad_contours = ax.contour(
        temp_grad_850.longitude,
        temp_grad_850.latitude,
        temp_grad_850,
        levels=TEMP_GRAD_850_CONTOUR_LEVELS,
        colors=SURFACE_TGRAD_CONTOUR_COLOR,
        linewidths=1.0,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(tempgrad_contours, inline=True, fontsize=7, fmt="%.0f")
    quiver = ax.quiver(
        snapshot_850.longitude.values[::LOW_LEVEL_WIND_STRIDE],
        snapshot_850.latitude.values[::LOW_LEVEL_WIND_STRIDE],
        snapshot_850["u_component_of_wind"].values[::LOW_LEVEL_WIND_STRIDE, ::LOW_LEVEL_WIND_STRIDE],
        snapshot_850["v_component_of_wind"].values[::LOW_LEVEL_WIND_STRIDE, ::LOW_LEVEL_WIND_STRIDE],
        transform=ccrs.PlateCarree(),
        color="black",
        scale=LOW_LEVEL_QUIVER_SCALE,
        width=0.0022,
        headwidth=3.0,
        headlength=4.2,
        pivot="mid",
        alpha=0.85,
        zorder=5,
    )
    legend_handles, legend_labels = add_region_outlines(ax)
    legend_handles.extend([
        plt.Line2D([0], [0], color=SURFACE_MSLP_CONTOUR_COLOR, linewidth=1.8),
        plt.Line2D([0], [0], color=SURFACE_TGRAD_CONTOUR_COLOR, linewidth=1.4),
        plt.Line2D([0], [0], color="black", linewidth=1.0),
    ])
    legend_labels.extend([
        "MSLP contours [hPa]",
        "850 hPa |grad T| magnitude contours [K (100 km)^-1]",
        "850 hPa wind vectors [m s^-1]",
    ])
    ax.legend(
        legend_handles,
        legend_labels,
        loc="lower left",
        frameon=True,
        facecolor="white",
        framealpha=0.92,
        edgecolor="0.70",
        fontsize=8,
    )
    ax.quiverkey(
        quiver,
        0.88,
        -0.10,
        LOW_LEVEL_REFERENCE_VECTOR,
        f"{LOW_LEVEL_REFERENCE_VECTOR:.0f} m s^-1",
        labelpos="E",
        coordinates="axes",
        fontproperties={"size": 8},
    )

    for target_ax, mappable, label in [
        (axes[0], jet_fill_300, f"300 hPa wind speed [m s^-1] (only >= {JET_ISOTACH_MIN_WIND_SPEED:.0f} shown)"),
        (axes[1], jet_fill_500, f"300 hPa wind speed [m s^-1] (only >= {JET_ISOTACH_MIN_WIND_SPEED:.0f} shown)"),
    ]:
        bbox = target_ax.get_position()
        cbar_ax = fig.add_axes([bbox.x0, 0.08, bbox.width, 0.032])
        cbar = fig.colorbar(mappable, cax=cbar_ax, orientation="horizontal")
        cbar.set_label(label)

    time_jst = analysis_time + pd.Timedelta(hours=9)
    fig.suptitle(
        f"Timing group {context['selected_group_id']} | forecast-funnel context | {time_role} | {analysis_time:%Y-%m-%d %H:%M UTC} / {time_jst:%Y-%m-%d %H:%M JST}",
        fontsize=13,
        y=0.98,
    )

    if SAVE_PLOTS:
        safe_role = time_role.replace(" ", "_").replace("/", "-")
        output_path = REVIEW_PLOT_DIR / f"{context['selected_group_id']}_{safe_role}_synoptic_funnel.png"
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        maybe_copy_to_drive(output_path, verbose=False)
    return fig


def plot_low_level_dynamics_figure(context: dict, time_role: str):
    role_lookup = {label: pd.Timestamp(timestamp) for label, timestamp in context["representative_time_specs"]}
    if time_role not in role_lookup:
        raise RuntimeError(
            f"Representative time role {time_role!r} is not available for timing group {context['selected_group_id']!r}."
        )
    analysis_time = role_lookup[time_role]
    era5_runtime_ds = get_era5_runtime_ds()

    snapshot_925 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["temperature", "u_component_of_wind", "v_component_of_wind"],
        domain=SURFACE_CONTEXT_DOMAIN,
        level=LOW_LEVEL_DYNAMICS_LEVEL_HPA,
    )
    divergence_925 = (compute_divergence_field(snapshot_925) * 1e5).rename("divergence_925_display")
    theta_925 = compute_potential_temperature_field(
        snapshot_925,
        pressure_hpa=LOW_LEVEL_DYNAMICS_LEVEL_HPA,
    )

    terrain_local = None
    terrain_fill = None
    terrain_field = get_surface_elevation_field_26()
    if terrain_field is not None:
        terrain_subset = terrain_field.sel(
            longitude=slice(SURFACE_CONTEXT_DOMAIN.lon_min, SURFACE_CONTEXT_DOMAIN.lon_max),
            latitude=slice(SURFACE_CONTEXT_DOMAIN.lat_max, SURFACE_CONTEXT_DOMAIN.lat_min),
        )
        terrain_local = align_field_to_target(terrain_subset, divergence_925)
        terrain_local = terrain_local.where(terrain_local > 0.0)

    fig, ax = plt.subplots(figsize=(10.8, 7.4), subplot_kw={"projection": ccrs.PlateCarree()})
    fig.subplots_adjust(top=0.84, bottom=0.20, left=0.06, right=0.98)

    ax.set_extent(
        [SURFACE_CONTEXT_DOMAIN.lon_min, SURFACE_CONTEXT_DOMAIN.lon_max, SURFACE_CONTEXT_DOMAIN.lat_min, SURFACE_CONTEXT_DOMAIN.lat_max],
        crs=ccrs.PlateCarree(),
    )
    ax.set_facecolor("white")
    if terrain_local is not None and np.isfinite(terrain_local.values).any():
        terrain_fill = ax.contourf(
            terrain_local.longitude,
            terrain_local.latitude,
            terrain_local,
            levels=LOW_LEVEL_TERRAIN_LEVELS,
            cmap="Greys",
            extend="max",
            alpha=0.74,
            transform=ccrs.PlateCarree(),
            zorder=0,
        )

    divergence_fill = ax.contourf(
        divergence_925.longitude,
        divergence_925.latitude,
        divergence_925,
        levels=LOW_LEVEL_DIVERGENCE_LEVELS,
        cmap="RdBu_r",
        extend="both",
        alpha=0.76,
        transform=ccrs.PlateCarree(),
        zorder=1,
    )
    theta_levels = rounded_contour_levels(theta_925, step=LOW_LEVEL_THETA_CONTOUR_STEP)
    theta_contours = ax.contour(
        theta_925.longitude,
        theta_925.latitude,
        theta_925,
        levels=theta_levels,
        colors="#b44d4d",
        linewidths=1.0,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    ax.clabel(theta_contours, inline=True, fontsize=7, fmt="%d")
    quiver = ax.quiver(
        snapshot_925.longitude.values[::LOW_LEVEL_DYNAMICS_WIND_STRIDE],
        snapshot_925.latitude.values[::LOW_LEVEL_DYNAMICS_WIND_STRIDE],
        snapshot_925["u_component_of_wind"].values[::LOW_LEVEL_DYNAMICS_WIND_STRIDE, ::LOW_LEVEL_DYNAMICS_WIND_STRIDE],
        snapshot_925["v_component_of_wind"].values[::LOW_LEVEL_DYNAMICS_WIND_STRIDE, ::LOW_LEVEL_DYNAMICS_WIND_STRIDE],
        transform=ccrs.PlateCarree(),
        color="black",
        scale=LOW_LEVEL_DYNAMICS_QUIVER_SCALE,
        width=0.0021,
        headwidth=3.0,
        headlength=4.2,
        pivot="mid",
        alpha=0.84,
        zorder=5,
    )

    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.85, edgecolor="black", zorder=6)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.35, edgecolor="black", zorder=6)
    gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.25, alpha=0.30, color="0.60")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    ax.set_title(
        f"Local low-level dynamics at {LOW_LEVEL_DYNAMICS_LEVEL_HPA:.0f} hPa",
        fontsize=12,
        pad=6,
    )

    legend_handles, legend_labels = add_region_outlines(ax)
    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.92),
        ncol=2,
        frameon=False,
    )
    ax.quiverkey(
        quiver,
        0.88,
        -0.10,
        LOW_LEVEL_DYNAMICS_REFERENCE_VECTOR,
        f"{LOW_LEVEL_DYNAMICS_REFERENCE_VECTOR:.0f} m s^-1",
        labelpos="E",
        coordinates="axes",
        fontproperties={"size": 8},
    )

    fig.text(
        0.5,
        0.14,
        "Shading: 925 hPa signed divergence [1e-5 s^-1] | Red contours: 925 hPa potential temperature theta [K] | Gray terrain: elevation [m] | Vectors: 925 hPa wind [m s^-1]",
        ha="center",
        va="center",
        fontsize=8.5,
        color="#444444",
    )
    div_cbar_ax = fig.add_axes([0.09, 0.07, 0.40, 0.035])
    div_cbar = fig.colorbar(divergence_fill, cax=div_cbar_ax, orientation="horizontal")
    div_cbar.set_label("925 hPa signed divergence [1e-5 s^-1]")
    if terrain_fill is not None:
        terrain_cbar_ax = fig.add_axes([0.56, 0.07, 0.28, 0.035])
        terrain_cbar = fig.colorbar(terrain_fill, cax=terrain_cbar_ax, orientation="horizontal")
        terrain_cbar.set_label("Terrain height [m]")
    else:
        ax.text(
            0.99,
            0.02,
            "Terrain background unavailable",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=8,
            color="#666666",
            bbox={"boxstyle": "round,pad=0.20", "facecolor": "white", "alpha": 0.82, "edgecolor": "0.78"},
        )

    time_jst = analysis_time + pd.Timedelta(hours=9)
    fig.suptitle(
        f"Timing group {context['selected_group_id']} | low-level dynamics | {time_role} | {analysis_time:%Y-%m-%d %H:%M UTC} / {time_jst:%Y-%m-%d %H:%M JST}",
        fontsize=13,
        y=0.98,
    )

    if SAVE_PLOTS:
        safe_role = time_role.replace(" ", "_").replace("/", "-")
        output_path = REVIEW_PLOT_DIR / f"{context['selected_group_id']}_{safe_role}_low_level_dynamics.png"
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        maybe_copy_to_drive(output_path, verbose=False)
    return fig


def plot_continuous_evolution_figure(context: dict):
    selected_group_timeseries_df = context["selected_group_timeseries_df"]
    selected_group_row = context["selected_group_row"]
    selected_profile_event_df = context["selected_profile_event_df"]
    selected_threshold_row = context["selected_threshold_row"]

    if selected_group_timeseries_df.empty:
        return None

    state_display_labels = {
        "offshore_jpcz_core": "Offshore support",
        "coastal_interaction": "Coastal support",
        "mixed_transition": "Mixed support",
        "weak_or_unclear": "Weak / unclear",
    }

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(11.8, 8.6),
        sharex=True,
        gridspec_kw={"height_ratios": [0.42, 1.0, 1.0]},
    )
    fig.subplots_adjust(top=0.84, bottom=0.16, left=0.10, right=0.96, hspace=0.14)

    state_ax, moisture_ax, divergence_ax = axes
    state_ax.set_ylim(0.0, 1.0)
    state_ax.set_yticks([])
    state_ax.set_ylabel("State")
    state_ax.set_title("Persistent support state through time", fontsize=10, loc="left")
    for spine_name in ["left", "right", "top"]:
        state_ax.spines[spine_name].set_visible(False)

    for _, row in selected_group_timeseries_df.iterrows():
        color = STATE_COLOR_MAP.get(str(row["continuous_regime_label"]), "#dddddd")
        state_ax.axvspan(
            row["time"],
            row["time"] + pd.Timedelta(hours=CONTINUOUS_TIME_STEP_HOURS),
            color=color,
            alpha=0.55,
            zorder=0,
        )

    regime_values = selected_group_timeseries_df["continuous_regime_label"].astype(str).tolist()
    segment_start = 0
    for segment_end in range(1, len(regime_values) + 1):
        segment_changed = segment_end == len(regime_values) or regime_values[segment_end] != regime_values[segment_start]
        if not segment_changed:
            continue
        segment_label = regime_values[segment_start]
        segment_start_time = pd.Timestamp(selected_group_timeseries_df.iloc[segment_start]["time"])
        segment_end_time = pd.Timestamp(selected_group_timeseries_df.iloc[segment_end - 1]["time"]) + pd.Timedelta(hours=CONTINUOUS_TIME_STEP_HOURS)
        segment_duration_hours = (segment_end_time - segment_start_time) / pd.Timedelta(hours=1)
        if segment_duration_hours >= 18:
            midpoint = segment_start_time + (segment_end_time - segment_start_time) / 2
            state_ax.text(
                midpoint,
                0.50,
                state_display_labels.get(segment_label, segment_label),
                ha="center",
                va="center",
                fontsize=8,
                color="black",
                bbox={"boxstyle": "round,pad=0.20", "facecolor": "white", "alpha": 0.60, "edgecolor": "none"},
            )
        segment_start = segment_end

    for event_row in selected_profile_event_df.itertuples(index=False):
        event_peak_time = pd.Timestamp(event_row.event_peak)
        for ax in axes:
            ax.axvline(event_peak_time, color="black", linestyle=":", linewidth=1.0, alpha=0.75)

    offshore_time = selected_group_row.get("first_offshore_support_time", pd.NaT)
    coastal_time = selected_group_row.get("first_coastal_support_time", pd.NaT)
    if pd.notna(offshore_time):
        for ax in axes:
            ax.axvline(pd.Timestamp(offshore_time), color="#4c78a8", linestyle="--", linewidth=1.3, alpha=0.95)
    if pd.notna(coastal_time):
        for ax in axes:
            ax.axvline(pd.Timestamp(coastal_time), color="#f58518", linestyle="--", linewidth=1.3, alpha=0.95)

    moisture_ax.plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["polygon_qflux_850_mean"],
        color="#1f3b73",
        linewidth=2.2,
        marker="o",
        markersize=3.5,
        label="Polygon moisture proxy",
    )
    moisture_ax.plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["coastal_qflux_850_mean"],
        color="#b85c00",
        linewidth=2.2,
        marker="o",
        markersize=3.5,
        label="Coastal-only moisture proxy",
    )
    moisture_ax.axhline(
        float(selected_threshold_row["polygon_qflux_min"]),
        color="#6f87b5",
        linestyle="--",
        linewidth=1.3,
        label="Polygon moisture threshold",
    )
    moisture_ax.axhline(
        float(selected_threshold_row["coastal_qflux_split"]),
        color="#d99859",
        linestyle="--",
        linewidth=1.3,
        label="Coastal moisture threshold",
    )
    moisture_ax.axhline(0.0, color="0.72", linewidth=0.9)
    moisture_ax.set_ylabel("850 hPa q × (-omega) [1e-3 Pa s^-1]")
    moisture_ax.grid(alpha=0.22)
    moisture_ax.legend(loc="upper left", ncol=2, fontsize=9, frameon=True, framealpha=0.88)

    divergence_ax.plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["polygon_div_925_mean"],
        color="#1f3b73",
        linewidth=2.2,
        marker="o",
        markersize=3.5,
        label="Polygon divergence",
    )
    divergence_ax.plot(
        selected_group_timeseries_df["time"],
        selected_group_timeseries_df["coastal_div_925_mean"],
        color="#b85c00",
        linewidth=2.2,
        marker="o",
        markersize=3.5,
        label="Coastal-only divergence",
    )
    divergence_ax.axhline(
        float(selected_threshold_row["polygon_div_max"]),
        color="#6f87b5",
        linestyle="--",
        linewidth=1.3,
        label="Polygon divergence threshold",
    )
    divergence_ax.axhline(
        float(selected_threshold_row["coastal_div_max"]),
        color="#d99859",
        linestyle="--",
        linewidth=1.3,
        label="Coastal divergence threshold",
    )
    divergence_ax.axhline(0.0, color="0.72", linewidth=0.9)
    divergence_ax.set_ylabel("925 hPa divergence [1e-5 s^-1]")
    divergence_ax.set_xlabel("Time")
    divergence_ax.grid(alpha=0.22)
    divergence_ax.legend(loc="upper left", ncol=2, fontsize=9, frameon=True, framealpha=0.88)

    state_handles = [
        Patch(facecolor=color, alpha=0.55, label=state_display_labels.get(label, label))
        for label, color in STATE_COLOR_MAP.items()
    ]
    line_handles = [
        plt.Line2D([0], [0], color="black", linestyle=":", linewidth=1.0, label="Event peak"),
        plt.Line2D([0], [0], color="#4c78a8", linestyle="--", linewidth=1.3, label="First offshore-support onset"),
        plt.Line2D([0], [0], color="#f58518", linestyle="--", linewidth=1.3, label="First coastal-support onset"),
    ]
    fig.legend(line_handles, [handle.get_label() for handle in line_handles], loc="upper center", bbox_to_anchor=(0.5, 0.93), ncol=3, frameon=False)
    fig.legend(state_handles, [handle.get_label() for handle in state_handles], loc="lower center", bbox_to_anchor=(0.5, 0.03), ncol=4, frameon=False)

    fig.suptitle(
        f"Timing group {context['selected_group_id']} | continuous evolution | {context['threshold_profile']} | {selected_group_row['continuous_spell_regime_path']}",
        fontsize=13,
        y=0.98,
    )
    return fig


def plot_cross_section_setup_figure(context: dict, time_role: str, start_lon=None, start_lat=None, end_lon=None, end_lat=None):
    role_lookup = {label: pd.Timestamp(timestamp) for label, timestamp in context["representative_time_specs"]}
    if time_role not in role_lookup:
        raise RuntimeError(
            f"Representative time role {time_role!r} is not available for timing group {context['selected_group_id']!r}."
        )
    analysis_time = role_lookup[time_role]
    era5_runtime_ds = get_era5_runtime_ds()

    snapshot_300 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["u_component_of_wind", "v_component_of_wind", "geopotential"],
        domain=JET_CONTEXT_DOMAIN,
        level=300,
    )
    snapshot_500 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["geopotential"],
        domain=JET_CONTEXT_DOMAIN,
        level=500,
    )
    surface_snapshot_jet = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["mean_sea_level_pressure"],
        domain=JET_CONTEXT_DOMAIN,
    )

    wind_speed_300 = compute_wind_speed_field(snapshot_300)
    jet_isotach_field = wind_speed_300.where(wind_speed_300 >= JET_ISOTACH_MIN_WIND_SPEED)
    z500 = compute_geopotential_height_field(snapshot_500)
    msl_hpa_jet = (surface_snapshot_jet["mean_sea_level_pressure"] / 100.0).rename("msl_hpa_jet")

    fig, ax = plt.subplots(figsize=(10.2, 6.9), subplot_kw={"projection": ccrs.PlateCarree()})
    jet_fill = ax.contourf(
        jet_isotach_field.longitude,
        jet_isotach_field.latitude,
        jet_isotach_field,
        levels=JET_ISOTACH_LEVELS,
        cmap="YlGnBu_r",
        extend="max",
        transform=ccrs.PlateCarree(),
    )
    configure_map_axis(ax, JET_CONTEXT_DOMAIN, title="Cross-section setup preview on 500 hPa height + jet isotachs + surface MSLP")
    z500_contours = ax.contour(
        z500.longitude,
        z500.latitude,
        z500,
        levels=rounded_contour_levels(z500, step=60.0),
        colors="#303030",
        linewidths=0.9,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(z500_contours, inline=True, fontsize=7, fmt="%d")
    msl_jet_contours = ax.contour(
        msl_hpa_jet.longitude,
        msl_hpa_jet.latitude,
        msl_hpa_jet,
        levels=rounded_contour_levels(msl_hpa_jet, step=4.0),
        colors=SURFACE_MSLP_CONTOUR_COLOR,
        linewidths=0.9,
        alpha=0.95,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(msl_jet_contours, inline=True, fontsize=6.5, fmt="%d")
    legend_handles, legend_labels = add_region_outlines(ax)

    valid_line = all(value is not None for value in [start_lon, start_lat, end_lon, end_lat])
    if valid_line:
        ax.plot(
            [start_lon, end_lon],
            [start_lat, end_lat],
            color="#7c3aed",
            linewidth=2.6,
            transform=ccrs.PlateCarree(),
            zorder=6,
        )
        ax.scatter([start_lon], [start_lat], color="#16a34a", s=60, transform=ccrs.PlateCarree(), zorder=7)
        ax.scatter([end_lon], [end_lat], color="#dc2626", s=60, transform=ccrs.PlateCarree(), zorder=7)
        legend_handles.extend([
            plt.Line2D([0], [0], color="#7c3aed", linewidth=2.6),
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#16a34a", markersize=8),
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#dc2626", markersize=8),
        ])
        legend_labels.extend([
            "Cross-section transect",
            "Start point",
            "End point",
        ])
    else:
        ax.text(
            0.02,
            0.02,
            "Enter start/end lon-lat in the Cross Sections or Review tab to preview the transect.",
            transform=ax.transAxes,
            fontsize=9,
            color="#374151",
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.88, "edgecolor": "0.75"},
        )

    time_jst = analysis_time + pd.Timedelta(hours=9)
    fig.suptitle(
        f"Timing group {context['selected_group_id']} | cross-section preview | {time_role} | {analysis_time:%Y-%m-%d %H:%M UTC} / {time_jst:%Y-%m-%d %H:%M JST}",
        fontsize=12,
        y=0.98,
    )
    legend_handles.append(plt.Line2D([0], [0], color=SURFACE_MSLP_CONTOUR_COLOR, linewidth=1.5))
    legend_labels.append("Surface MSLP contours [hPa]")
    ax.legend(legend_handles, legend_labels, loc="lower left", frameon=True, facecolor="white", framealpha=0.92, edgecolor="0.70", fontsize=8)
    cbar_ax = fig.add_axes([0.14, 0.08, 0.72, 0.035])
    cbar = fig.colorbar(jet_fill, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(f"300 hPa wind speed [m s^-1] (only >= {JET_ISOTACH_MIN_WIND_SPEED:.0f} shown)")
    return fig


print("Notebook 26 interactive review helpers are ready")
print(f"- Review scaffold: {TIMING_GROUP_REVIEW_SCAFFOLD_PATH}")
print(f"- Threshold profiles available: {', '.join(PROFILE_ORDER)}")
print(f"- Timing groups available for manual review: {timing_group_review_scaffold_df['catalog_spell_id'].nunique()}")


In [ ]:

required_globals = [
    "timing_group_review_scaffold_df",
    "PROFILE_ORDER",
    "build_group_options",
    "choose_default_group_id",
    "get_group_context",
    "plot_event_firstlook_figure",
    "render_saved_imagery_for_event",
    "plot_synoptic_context_figure",
    "plot_low_level_dynamics_figure",
    "plot_continuous_evolution_figure",
    "plot_cross_section_setup_figure",
    "_save_review_scaffold",
    "_normalize_text",
    "_split_csv_tokens",
    "_split_pipe_tokens",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the Notebook 26 interactive helper cell first. Missing globals: {missing_globals}")

ui_state = {
    "context": None,
    "suspend": False,
    "event_checkboxes": [],
    "time_checkboxes": [],
}

profile_dropdown = widgets.Dropdown(
    options=PROFILE_ORDER,
    value=REVIEW_THRESHOLD_PROFILE if REVIEW_THRESHOLD_PROFILE in PROFILE_ORDER else PROFILE_ORDER[0],
    description="Profile",
    layout=widgets.Layout(width="260px"),
)
group_dropdown = widgets.Dropdown(
    options=[],
    description="Timing group",
    layout=widgets.Layout(width="720px"),
)
prev_group_button = widgets.Button(description="Prev group", icon="arrow-left")
next_group_button = widgets.Button(description="Next group", icon="arrow-right")
group_jump_text = widgets.Text(
    value="",
    placeholder="catalog_36h_134",
    description="Jump ID",
    layout=widgets.Layout(width="280px"),
)
group_jump_button = widgets.Button(description="Go", icon="search")

event_dropdown = widgets.Dropdown(
    options=[],
    description="Event",
    layout=widgets.Layout(width="620px"),
)
prev_event_button = widgets.Button(description="Prev event", icon="arrow-left")
next_event_button = widgets.Button(description="Next event", icon="arrow-right")
time_dropdown = widgets.Dropdown(
    options=[],
    description="Time",
    layout=widgets.Layout(width="520px"),
)
selection_status_html = widgets.HTML()
review_header_html = widgets.HTML()
save_status_html = widgets.HTML()

overview_output = widgets.Output()
event_maps_output = widgets.Output()
imagery_output = widgets.Output()
synoptic_output = widgets.Output()
low_level_output = widgets.Output()
continuous_output = widgets.Output()
cross_section_output = widgets.Output()

manual_transition_verified_widget = widgets.Dropdown(
    options=["", "yes", "no", "uncertain"],
    description="Verified",
    layout=widgets.Layout(width="260px"),
)
manual_transition_type_widget = widgets.Dropdown(
    options=["", "offshore_to_coastal", "coastal_to_offshore", "offshore_only", "coastal_only", "mixed_or_oscillating", "weak_only", "other"],
    description="Type",
    layout=widgets.Layout(width="320px"),
)
manual_transition_confidence_widget = widgets.Dropdown(
    options=["", "low", "medium", "high"],
    description="Confidence",
    layout=widgets.Layout(width="250px"),
)
manual_offshore_onset_jst_widget = widgets.Text(description="Offshore JST", layout=widgets.Layout(width="260px"))
manual_coastal_onset_jst_widget = widgets.Text(description="Coastal JST", layout=widgets.Layout(width="260px"))
manual_transition_lag_hours_widget = widgets.Text(description="Lag [h]", layout=widgets.Layout(width="180px"))
satellite_supports_transition_widget = widgets.Dropdown(
    options=["", "yes", "no", "mixed"],
    description="Satellite",
    layout=widgets.Layout(width="260px"),
)
forecast_funnel_supports_transition_widget = widgets.Dropdown(
    options=["", "yes", "no", "mixed"],
    description="Funnel",
    layout=widgets.Layout(width="250px"),
)
coastal_impact_visible_widget = widgets.Dropdown(
    options=["", "yes", "no", "unclear"],
    description="Impact",
    layout=widgets.Layout(width="240px"),
)
coastal_impact_region_widget = widgets.Text(description="Region", layout=widgets.Layout(width="360px"))
cross_section_priority_widget = widgets.Dropdown(
    options=["", "none", "low", "medium", "high"],
    description="X-sec prio",
    layout=widgets.Layout(width="250px"),
)
cross_section_start_lon_widget = widgets.Text(description="Start lon", layout=widgets.Layout(width="180px"))
cross_section_start_lat_widget = widgets.Text(description="Start lat", layout=widgets.Layout(width="180px"))
cross_section_end_lon_widget = widgets.Text(description="End lon", layout=widgets.Layout(width="180px"))
cross_section_end_lat_widget = widgets.Text(description="End lat", layout=widgets.Layout(width="180px"))
manual_notes_widget = widgets.Textarea(
    value="",
    description="Notes",
    layout=widgets.Layout(width="100%", height="160px"),
)
reviewed_event_box = widgets.VBox()
reviewed_time_box = widgets.VBox()
reviewed_event_header = widgets.HTML("<b>Reviewed events</b>")
reviewed_time_header = widgets.HTML("<b>Reviewed representative times</b>")
review_guidance_html = widgets.HTML(
    "<b>Cross-section recommendation:</b> use the 500 hPa preview in the Cross Sections tab for broad placement and the Low-Level Dynamics tab for local refinement. "
    "Typed endpoints or presets will still be more stable here than click-to-place endpoints inside the tabbed widget layout."
)
save_button = widgets.Button(description="Save review row", button_style="success", icon="save")
reload_button = widgets.Button(description="Reload saved row", icon="refresh")

review_left_box = widgets.VBox([
    widgets.HTML("<b>Transition assessment</b>"),
    widgets.HBox([manual_transition_verified_widget, manual_transition_type_widget, manual_transition_confidence_widget]),
    widgets.HBox([manual_offshore_onset_jst_widget, manual_coastal_onset_jst_widget, manual_transition_lag_hours_widget]),
    widgets.HBox([satellite_supports_transition_widget, forecast_funnel_supports_transition_widget]),
    widgets.HBox([coastal_impact_visible_widget, coastal_impact_region_widget]),
])
review_right_box = widgets.VBox([
    widgets.HTML("<b>Cross-section setup</b>"),
    cross_section_priority_widget,
    widgets.HBox([cross_section_start_lon_widget, cross_section_start_lat_widget]),
    widgets.HBox([cross_section_end_lon_widget, cross_section_end_lat_widget]),
    review_guidance_html,
])
review_form_box = widgets.VBox([
    review_header_html,
    widgets.HBox([review_left_box, review_right_box], layout=widgets.Layout(align_items="flex-start", justify_content="space-between")),
    widgets.HBox([
        widgets.VBox([reviewed_event_header, reviewed_event_box], layout=widgets.Layout(width="49%")),
        widgets.VBox([reviewed_time_header, reviewed_time_box], layout=widgets.Layout(width="49%")),
    ], layout=widgets.Layout(align_items="flex-start", justify_content="space-between")),
    manual_notes_widget,
    widgets.HBox([save_button, reload_button, save_status_html]),
])

dashboard_tabs = widgets.Tab(children=[
    overview_output,
    event_maps_output,
    imagery_output,
    synoptic_output,
    low_level_output,
    continuous_output,
    cross_section_output,
    review_form_box,
])
for idx, title in enumerate(["Overview", "Event Maps", "Imagery", "Synoptic", "Low-Level Dynamics", "Continuous", "Cross Sections", "Review"]):
    dashboard_tabs.set_title(idx, title)


def set_dropdown_if_valid(widget, value):
    valid_values = []
    for option in widget.options:
        if isinstance(option, tuple):
            valid_values.append(option[1])
        else:
            valid_values.append(option)
    widget.value = value if value in valid_values else ""


def mark_review_dirty(*_):
    save_status_html.value = '<span style="color:#b45309;"><b>Unsaved changes</b></span>'


def current_group_values() -> list[str]:
    return [value for _, value in group_dropdown.options]


def current_event_values() -> list[int]:
    return [value for _, value in event_dropdown.options]


def build_event_options(context: dict) -> list[tuple[str, int]]:
    options = []
    for event_row in context["selected_profile_event_df"].itertuples(index=False):
        peak_utc = pd.Timestamp(event_row.event_peak)
        label = (
            f"event {int(event_row.event_index):03d} | {peak_utc:%Y-%m-%d %H:%M UTC} | "
            f"default {event_row.default_threshold_label} | {context['threshold_profile']} {event_row.selected_profile_event_label}"
        )
        options.append((label, int(event_row.event_index)))
    return options


def build_time_options(context: dict) -> list[tuple[str, str]]:
    options = []
    for label, timestamp in context["representative_time_specs"]:
        utc_time = pd.Timestamp(timestamp)
        jst_time = utc_time + pd.Timedelta(hours=9)
        options.append((f"{label} | {utc_time:%Y-%m-%d %H:%M UTC} / {jst_time:%Y-%m-%d %H:%M JST}", label))
    return options


def update_group_controls(profile: str, preferred_group_id: str | None = None):
    options = build_group_options(profile)
    if not options:
        raise RuntimeError(f"No timing-group options were available for profile {profile!r}.")
    group_values = [value for _, value in options]
    if preferred_group_id not in group_values:
        preferred_group_id = choose_default_group_id(profile)
    if preferred_group_id not in group_values:
        preferred_group_id = group_values[0]

    ui_state["suspend"] = True
    group_dropdown.options = options
    group_dropdown.value = preferred_group_id
    group_jump_text.value = preferred_group_id
    ui_state["suspend"] = False


def update_event_controls(context: dict, preferred_event_index: int | None = None):
    options = build_event_options(context)
    if not options:
        raise RuntimeError(f"No events were available in timing group {context['selected_group_id']!r}.")
    event_values = [value for _, value in options]
    if preferred_event_index not in event_values:
        preferred_event_index = event_values[0]

    ui_state["suspend"] = True
    event_dropdown.options = options
    event_dropdown.value = preferred_event_index
    ui_state["suspend"] = False


def update_time_controls(context: dict, preferred_time_role: str | None = None):
    options = build_time_options(context)
    ui_state["suspend"] = True
    if not options:
        time_dropdown.options = [("No representative time available", "")]
        time_dropdown.value = ""
        time_dropdown.disabled = True
    else:
        values = [value for _, value in options]
        if preferred_time_role not in values:
            preferred_time_role = values[0]
        time_dropdown.options = options
        time_dropdown.value = preferred_time_role
        time_dropdown.disabled = False
    ui_state["suspend"] = False


def update_nav_button_state():
    group_values = current_group_values()
    current_group = group_dropdown.value
    prev_group_button.disabled = not group_values or group_values.index(current_group) == 0
    next_group_button.disabled = not group_values or group_values.index(current_group) == len(group_values) - 1

    event_values = current_event_values()
    current_event = event_dropdown.value
    prev_event_button.disabled = not event_values or event_values.index(current_event) == 0
    next_event_button.disabled = not event_values or event_values.index(current_event) == len(event_values) - 1


def update_selection_status(context: dict):
    selected_group_row = context["selected_group_row"]
    current_event_label = next((label for label, value in event_dropdown.options if value == event_dropdown.value), "")
    current_time_label = next((label for label, value in time_dropdown.options if value == time_dropdown.value), "No representative time selected")
    selection_status_html.value = (
        f"<b>{context['selected_group_id']}</b> | {context['threshold_profile']} | "
        f"path: <code>{selected_group_row['continuous_spell_regime_path']}</code> | "
        f"events: {int(selected_group_row['event_count'])}<br>"
        f"<span style='color:#555;'>Current event:</span> {current_event_label}<br>"
        f"<span style='color:#555;'>Current representative time:</span> {current_time_label}"
    )


def render_overview(context: dict):
    overview_output.clear_output(wait=True)
    with overview_output:
        header_md = (
            f"## Timing group `{context['selected_group_id']}` | profile `{context['threshold_profile']}`\n\n"
            f"Continuous path: `{context['selected_group_row']['continuous_spell_regime_path']}`\n\n"
            f"Clear sequence: `{context['selected_group_row']['continuous_clear_sequence']}`"
        )
        display(Markdown(header_md))
        print("Timing-group header")
        display(context["selected_group_header_df"])
        print("Threshold values active in this review")
        display(context["threshold_display_df"])
        print("How this timing group behaves across all three threshold profiles")
        display(context["selected_group_all_profiles_df"][[
            "threshold_profile",
            "continuous_spell_regime_path",
            "continuous_clear_sequence",
            "event_count",
            "offshore_to_coastal_lag_hours",
            "coastal_to_offshore_lag_hours",
        ]])
        print("Event membership table")
        display(context["selected_event_display_df"])
        if not context["representative_time_table_df"].empty:
            print("Representative synoptic times")
            display(context["representative_time_table_df"])
        else:
            print("No representative synoptic times were available for this timing group.")


def render_event_maps(context: dict):
    event_maps_output.clear_output(wait=True)
    with event_maps_output:
        fig = plot_event_firstlook_figure(context, int(event_dropdown.value))
        display(fig)
        plt.close(fig)


def render_imagery(context: dict):
    imagery_output.clear_output(wait=True)
    with imagery_output:
        render_saved_imagery_for_event(context, int(event_dropdown.value))


def render_synoptic(context: dict):
    synoptic_output.clear_output(wait=True)
    with synoptic_output:
        if not context["representative_time_specs"]:
            print("No representative synoptic times are available for this timing group.")
            return
        if not _normalize_text(time_dropdown.value):
            print("Choose a representative time to render the forecast-funnel context.")
            return
        fig = plot_synoptic_context_figure(context, str(time_dropdown.value))
        display(fig)
        plt.close(fig)


def render_synoptic_placeholder(context: dict):
    synoptic_output.clear_output(wait=True)
    with synoptic_output:
        if not context["representative_time_specs"]:
            print("No representative synoptic times are available for this timing group.")
        else:
            chosen_label = next((label for label, value in time_dropdown.options if value == time_dropdown.value), str(time_dropdown.value))
            print("Synoptic tab is ready.")
            print(f"Selected representative time: {chosen_label}")
            print("Open the Synoptic tab or use the current selection to render the forecast-funnel panel.")


def render_low_level(context: dict):
    low_level_output.clear_output(wait=True)
    with low_level_output:
        if not context["representative_time_specs"]:
            print("No representative low-level times are available for this timing group.")
            return
        if not _normalize_text(time_dropdown.value):
            print("Choose a representative time to render the low-level dynamics panel.")
            return
        fig = plot_low_level_dynamics_figure(context, str(time_dropdown.value))
        display(fig)
        plt.close(fig)


def render_low_level_placeholder(context: dict):
    low_level_output.clear_output(wait=True)
    with low_level_output:
        if not context["representative_time_specs"]:
            print("No representative low-level times are available for this timing group.")
        else:
            chosen_label = next((label for label, value in time_dropdown.options if value == time_dropdown.value), str(time_dropdown.value))
            print("Low-Level Dynamics tab is ready.")
            print(f"Selected representative time: {chosen_label}")
            print("Open the Low-Level Dynamics tab to render the terrain-aware divergence / wind / theta panel.")


def render_continuous(context: dict):
    continuous_output.clear_output(wait=True)
    with continuous_output:
        fig = plot_continuous_evolution_figure(context)
        if fig is None:
            print("No labeled continuous timing-group time series were found for this group/profile.")
            return
        display(fig)
        plt.close(fig)


def _float_or_none(value):
    text = _normalize_text(value)
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def render_cross_sections(context: dict):
    cross_section_output.clear_output(wait=True)
    with cross_section_output:
        display(Markdown(
            "**Cross-section setup**  \
Use the 500 hPa height + jet + surface-MSLP preview here for the broad transect placement, then use the **Low-Level Dynamics** tab to refine the local line against terrain, low-level divergence, wind, and theta.  \
The full PV / jet-normal / isentropic vertical cross-section plots are still the next build layer."
        ))
        selected_time_label = _normalize_text(time_dropdown.value)
        if not selected_time_label and context["representative_time_specs"]:
            selected_time_label = str(context["representative_time_specs"][0][0])
        current_event_index = event_dropdown.value if event_dropdown.value is not None else "none selected"
        summary_df = pd.DataFrame([
            {
                "priority": _normalize_text(cross_section_priority_widget.value, default=""),
                "selected_representative_time": selected_time_label,
                "selected_event_index": current_event_index,
                "start_lon": _normalize_text(cross_section_start_lon_widget.value),
                "start_lat": _normalize_text(cross_section_start_lat_widget.value),
                "end_lon": _normalize_text(cross_section_end_lon_widget.value),
                "end_lat": _normalize_text(cross_section_end_lat_widget.value),
            }
        ])
        display(summary_df)
        if not selected_time_label:
            print("No representative synoptic time is available for this timing group yet.")
            return
        fig = plot_cross_section_setup_figure(
            context,
            time_role=selected_time_label,
            start_lon=_float_or_none(cross_section_start_lon_widget.value),
            start_lat=_float_or_none(cross_section_start_lat_widget.value),
            end_lon=_float_or_none(cross_section_end_lon_widget.value),
            end_lat=_float_or_none(cross_section_end_lat_widget.value),
        )
        display(fig)
        plt.close(fig)


def render_cross_section_placeholder(context: dict):
    cross_section_output.clear_output(wait=True)
    with cross_section_output:
        print("Cross Sections tab is ready.")
        print("Open the Cross Sections tab to preview or edit the saved transect endpoints for this timing group.")


def sync_review_form(context: dict):
    scaffold_row = context["scaffold_row"]
    review_header_html.value = (
        f"<b>Review row:</b> {context['selected_group_id']} | {context['threshold_profile']} | "
        f"path <code>{context['selected_group_row']['continuous_spell_regime_path']}</code><br>"
        f"<span style='color:#555;'>Event-level sequence:</span> {_normalize_text(scaffold_row.get('event_level_sequence'))}"
    )

    set_dropdown_if_valid(manual_transition_verified_widget, _normalize_text(scaffold_row.get("manual_transition_verified")))
    set_dropdown_if_valid(manual_transition_type_widget, _normalize_text(scaffold_row.get("manual_transition_type")))
    set_dropdown_if_valid(manual_transition_confidence_widget, _normalize_text(scaffold_row.get("manual_transition_confidence")))
    manual_offshore_onset_jst_widget.value = _normalize_text(scaffold_row.get("manual_offshore_onset_jst"))
    manual_coastal_onset_jst_widget.value = _normalize_text(scaffold_row.get("manual_coastal_onset_jst"))
    manual_transition_lag_hours_widget.value = _normalize_text(scaffold_row.get("manual_transition_lag_hours"))
    set_dropdown_if_valid(satellite_supports_transition_widget, _normalize_text(scaffold_row.get("satellite_supports_transition")))
    set_dropdown_if_valid(forecast_funnel_supports_transition_widget, _normalize_text(scaffold_row.get("forecast_funnel_supports_transition")))
    set_dropdown_if_valid(coastal_impact_visible_widget, _normalize_text(scaffold_row.get("coastal_impact_visible")))
    coastal_impact_region_widget.value = _normalize_text(scaffold_row.get("coastal_impact_region"))
    set_dropdown_if_valid(cross_section_priority_widget, _normalize_text(scaffold_row.get("cross_section_priority")))
    cross_section_start_lon_widget.value = _normalize_text(scaffold_row.get("cross_section_start_lon"))
    cross_section_start_lat_widget.value = _normalize_text(scaffold_row.get("cross_section_start_lat"))
    cross_section_end_lon_widget.value = _normalize_text(scaffold_row.get("cross_section_end_lon"))
    cross_section_end_lat_widget.value = _normalize_text(scaffold_row.get("cross_section_end_lat"))
    manual_notes_widget.value = _normalize_text(scaffold_row.get("manual_notes"))

    checked_event_indices = _split_csv_tokens(scaffold_row.get("reviewed_event_indices", ""))
    checked_time_roles = _split_pipe_tokens(scaffold_row.get("reviewed_time_roles", ""))

    event_checkboxes = []
    for event_row in context["selected_profile_event_df"].itertuples(index=False):
        peak_utc = pd.Timestamp(event_row.event_peak)
        cb = widgets.Checkbox(
            value=str(int(event_row.event_index)) in checked_event_indices,
            description=(
                f"event {int(event_row.event_index):03d} | {peak_utc:%Y-%m-%d %H:%M UTC} | "
                f"{event_row.selected_profile_event_label}"
            ),
            indent=False,
            layout=widgets.Layout(width="100%"),
        )
        cb.observe(mark_review_dirty, names="value")
        event_checkboxes.append(cb)
    reviewed_event_box.children = tuple(event_checkboxes)
    ui_state["event_checkboxes"] = event_checkboxes

    time_checkboxes = []
    for label, timestamp in context["representative_time_specs"]:
        utc_time = pd.Timestamp(timestamp)
        jst_time = utc_time + pd.Timedelta(hours=9)
        cb = widgets.Checkbox(
            value=label in checked_time_roles,
            description=f"{label} | {utc_time:%Y-%m-%d %H:%M UTC} / {jst_time:%Y-%m-%d %H:%M JST}",
            indent=False,
            layout=widgets.Layout(width="100%"),
        )
        cb.observe(mark_review_dirty, names="value")
        time_checkboxes.append(cb)
    reviewed_time_box.children = tuple(time_checkboxes)
    ui_state["time_checkboxes"] = time_checkboxes

    save_status_html.value = '<span style="color:#166534;"><b>Review row loaded</b></span>'


def render_group_context(context: dict, *, preserve_event_index: int | None = None, preserve_time_role: str | None = None):
    ui_state["context"] = context
    update_event_controls(context, preferred_event_index=preserve_event_index)
    update_time_controls(context, preferred_time_role=preserve_time_role)
    update_nav_button_state()
    update_selection_status(context)
    render_overview(context)
    render_event_maps(context)
    render_imagery(context)
    render_continuous(context)
    sync_review_form(context)
    if dashboard_tabs.selected_index == 3:
        render_synoptic(context)
    else:
        render_synoptic_placeholder(context)
    if dashboard_tabs.selected_index == 4:
        render_low_level(context)
    else:
        render_low_level_placeholder(context)
    if dashboard_tabs.selected_index == 6:
        render_cross_sections(context)
    else:
        render_cross_section_placeholder(context)


def load_current_group_context(*, preserve_event_index: int | None = None, preserve_time_role: str | None = None):
    context = get_group_context(str(profile_dropdown.value), str(group_dropdown.value))
    render_group_context(context, preserve_event_index=preserve_event_index, preserve_time_role=preserve_time_role)


def on_profile_change(change):
    if ui_state["suspend"] or change.get("name") != "value":
        return
    update_group_controls(str(change["new"]))
    load_current_group_context()


def on_group_change(change):
    if ui_state["suspend"] or change.get("name") != "value" or change["new"] is None:
        return
    ui_state["suspend"] = True
    group_jump_text.value = str(change["new"])
    ui_state["suspend"] = False
    load_current_group_context()


def on_event_change(change):
    if ui_state["suspend"] or change.get("name") != "value" or ui_state["context"] is None:
        return
    update_nav_button_state()
    update_selection_status(ui_state["context"])
    render_event_maps(ui_state["context"])
    render_imagery(ui_state["context"])
    if dashboard_tabs.selected_index == 6:
        render_cross_sections(ui_state["context"])


def on_time_change(change):
    if ui_state["suspend"] or change.get("name") != "value" or ui_state["context"] is None:
        return
    update_selection_status(ui_state["context"])
    if dashboard_tabs.selected_index == 3:
        render_synoptic(ui_state["context"])
    else:
        render_synoptic_placeholder(ui_state["context"])
    if dashboard_tabs.selected_index == 4:
        render_low_level(ui_state["context"])
    else:
        render_low_level_placeholder(ui_state["context"])
    if dashboard_tabs.selected_index == 6:
        render_cross_sections(ui_state["context"])


def on_group_prev_clicked(_):
    values = current_group_values()
    current_index = values.index(group_dropdown.value)
    if current_index > 0:
        group_dropdown.value = values[current_index - 1]


def on_group_next_clicked(_):
    values = current_group_values()
    current_index = values.index(group_dropdown.value)
    if current_index < len(values) - 1:
        group_dropdown.value = values[current_index + 1]


def on_event_prev_clicked(_):
    values = current_event_values()
    current_index = values.index(event_dropdown.value)
    if current_index > 0:
        event_dropdown.value = values[current_index - 1]


def on_event_next_clicked(_):
    values = current_event_values()
    current_index = values.index(event_dropdown.value)
    if current_index < len(values) - 1:
        event_dropdown.value = values[current_index + 1]


def on_group_jump_clicked(_):
    requested_group_id = _normalize_text(group_jump_text.value)
    if not requested_group_id:
        return
    if requested_group_id not in current_group_values():
        save_status_html.value = f'<span style="color:#b91c1c;"><b>{requested_group_id}</b> is not available for profile {profile_dropdown.value}.</span>'
        return
    group_dropdown.value = requested_group_id


def on_tab_change(change):
    if change.get("name") != "selected_index" or ui_state["context"] is None:
        return
    if change["new"] == 3:
        render_synoptic(ui_state["context"])
    elif change["new"] == 4:
        render_low_level(ui_state["context"])
    elif change["new"] == 6:
        render_cross_sections(ui_state["context"])


def collect_review_updates() -> dict:
    reviewed_event_indices = ", ".join(
        str(int(event_row.event_index))
        for event_row, checkbox in zip(ui_state["context"]["selected_profile_event_df"].itertuples(index=False), ui_state["event_checkboxes"])
        if checkbox.value
    )
    reviewed_time_roles = " | ".join(
        label
        for (label, _), checkbox in zip(ui_state["context"]["representative_time_specs"], ui_state["time_checkboxes"])
        if checkbox.value
    )
    return {
        "manual_transition_verified": _normalize_text(manual_transition_verified_widget.value),
        "manual_transition_type": _normalize_text(manual_transition_type_widget.value),
        "manual_transition_confidence": _normalize_text(manual_transition_confidence_widget.value),
        "manual_offshore_onset_jst": _normalize_text(manual_offshore_onset_jst_widget.value),
        "manual_coastal_onset_jst": _normalize_text(manual_coastal_onset_jst_widget.value),
        "manual_transition_lag_hours": _normalize_text(manual_transition_lag_hours_widget.value),
        "satellite_supports_transition": _normalize_text(satellite_supports_transition_widget.value),
        "forecast_funnel_supports_transition": _normalize_text(forecast_funnel_supports_transition_widget.value),
        "reviewed_event_indices": reviewed_event_indices,
        "reviewed_time_roles": reviewed_time_roles,
        "cross_section_priority": _normalize_text(cross_section_priority_widget.value),
        "cross_section_start_lon": _normalize_text(cross_section_start_lon_widget.value),
        "cross_section_start_lat": _normalize_text(cross_section_start_lat_widget.value),
        "cross_section_end_lon": _normalize_text(cross_section_end_lon_widget.value),
        "cross_section_end_lat": _normalize_text(cross_section_end_lat_widget.value),
        "coastal_impact_visible": _normalize_text(coastal_impact_visible_widget.value),
        "coastal_impact_region": _normalize_text(coastal_impact_region_widget.value),
        "manual_notes": _normalize_text(manual_notes_widget.value),
    }


def on_save_clicked(_):
    context = ui_state["context"]
    if context is None:
        return
    mask = (
        timing_group_review_scaffold_df["catalog_spell_id"].astype(str) == context["selected_group_id"]
    ) & (
        timing_group_review_scaffold_df["threshold_profile"] == context["threshold_profile"]
    )
    if not mask.any():
        raise RuntimeError(f"No scaffold row found for timing group {context['selected_group_id']!r} and profile {context['threshold_profile']!r}.")
    row_index = timing_group_review_scaffold_df.index[mask][0]
    updates = collect_review_updates()
    for column_name, value in updates.items():
        timing_group_review_scaffold_df.at[row_index, column_name] = value
    _save_review_scaffold(timing_group_review_scaffold_df)
    ui_state["context"]["scaffold_row"] = timing_group_review_scaffold_df.loc[row_index].copy()
    save_status_html.value = (
        f'<span style="color:#166534;"><b>Saved</b> {context["selected_group_id"]} / {context["threshold_profile"]} '
        f'to <code>{TIMING_GROUP_REVIEW_SCAFFOLD_PATH.name}</code></span>'
    )


def on_reload_clicked(_):
    context = ui_state["context"]
    if context is None:
        return
    refreshed_row = timing_group_review_scaffold_df.loc[
        (timing_group_review_scaffold_df["catalog_spell_id"].astype(str) == context["selected_group_id"])
        & (timing_group_review_scaffold_df["threshold_profile"] == context["threshold_profile"])
    ].iloc[0].copy()
    ui_state["context"]["scaffold_row"] = refreshed_row
    sync_review_form(ui_state["context"])


for widget in [
    manual_transition_verified_widget,
    manual_transition_type_widget,
    manual_transition_confidence_widget,
    manual_offshore_onset_jst_widget,
    manual_coastal_onset_jst_widget,
    manual_transition_lag_hours_widget,
    satellite_supports_transition_widget,
    forecast_funnel_supports_transition_widget,
    coastal_impact_visible_widget,
    coastal_impact_region_widget,
    cross_section_priority_widget,
    cross_section_start_lon_widget,
    cross_section_start_lat_widget,
    cross_section_end_lon_widget,
    cross_section_end_lat_widget,
    manual_notes_widget,
]:
    widget.observe(mark_review_dirty, names="value")


def on_cross_section_field_change(change):
    if ui_state["suspend"] or change.get("name") != "value" or ui_state["context"] is None:
        return
    if dashboard_tabs.selected_index == 6:
        render_cross_sections(ui_state["context"])


for widget in [
    cross_section_priority_widget,
    cross_section_start_lon_widget,
    cross_section_start_lat_widget,
    cross_section_end_lon_widget,
    cross_section_end_lat_widget,
]:
    widget.observe(on_cross_section_field_change, names="value")

profile_dropdown.observe(on_profile_change, names="value")
group_dropdown.observe(on_group_change, names="value")
event_dropdown.observe(on_event_change, names="value")
time_dropdown.observe(on_time_change, names="value")
dashboard_tabs.observe(on_tab_change, names="selected_index")
prev_group_button.on_click(on_group_prev_clicked)
next_group_button.on_click(on_group_next_clicked)
prev_event_button.on_click(on_event_prev_clicked)
next_event_button.on_click(on_event_next_clicked)
group_jump_button.on_click(on_group_jump_clicked)
save_button.on_click(on_save_clicked)
reload_button.on_click(on_reload_clicked)

control_row_1 = widgets.HBox([
    profile_dropdown,
    prev_group_button,
    next_group_button,
    group_dropdown,
], layout=widgets.Layout(align_items="center", justify_content="flex-start"))
control_row_2 = widgets.HBox([
    group_jump_text,
    group_jump_button,
    prev_event_button,
    next_event_button,
    event_dropdown,
    time_dropdown,
], layout=widgets.Layout(align_items="center", justify_content="flex-start"))
dashboard = widgets.VBox([
    widgets.HTML("<h3 style='margin:0;'>Timing-group manual review dashboard</h3>"),
    control_row_1,
    control_row_2,
    selection_status_html,
    dashboard_tabs,
])

update_group_controls(str(profile_dropdown.value), preferred_group_id=choose_default_group_id(str(profile_dropdown.value)))
load_current_group_context()
display(dashboard)
